# Take-home Task: Job-Posting NLP Pipeline

## Overview

Build an NLP pipeline that processes job-posting data to extract insights about roles, skills, and job-market patterns.

The task assesses practical experience with:

- Text preprocessing and cleaning
- Embeddings and semantic similarity
- Text classification
- Basic retrieval-augmented generation (RAG)
- Evaluation methodology

**Expected effort:** approximately **6–8 hours**.
We value clean, well-reasoned code over a polished product. Document your decisions and trade-offs.

---

## Data

**Source:** [LinkedIn Job Postings – Kaggle](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings)  
A pre-filtered sample of ~4,000 tech/data job postings is provided.

---

## Requirements

Complete the following four tasks in order. Each builds on the previous.

| Task | What | Deliverable |
|------|------|-------------|
| 1. Text preprocessing | Clean and prepare job descriptions for downstream NLP | Code + brief explanation of choices |
| 2. Role classification | Classify postings into role categories using the description text | Trained model + evaluation |
| 3. Embeddings & similarity | Embed job postings and implement semantic search | Working search function |
| 4. Evaluation & reflection | Assess quality and discuss limitations | Written analysis |

---

## 1. Text preprocessing

Clean and prepare the `description` field for downstream NLP tasks.

At minimum:

1. Explore the data: look at description length, missing values, duplicates, and any quality issues.
2. Implement a preprocessing pipeline that produces clean text suitable for embedding and classification.
3. Document your choices (what you removed, what you kept, and why).

Things to consider:
- HTML/markdown artefacts
- Boilerplate text (e.g. equal-opportunity statements)
- Very short or empty descriptions
- Duplicate content
- How preprocessing choices affect downstream tasks differently

---

## 2. Role classification

Build a classifier that predicts the **role category** of a job posting from its description text.

### Setup

The `role_category` column contains a pre-assigned label derived by keyword-matching on the `title` field:

| Category | Count |
|----------|------:|
| Software Engineer | 1,739 |
| Data Analyst | 890 |
| Data Scientist | 416 |
| DevOps / Cloud Engineer | 411 |
| Data Engineer | 401 |
| Machine Learning Engineer | 159 |
| Other | 56 |

Your classifier should predict `role_category` using only the `description` field (not the title).

### Task

1. Create train/test splits.
2. Train at least **two approaches** and compare them. For example:
   - TF-IDF + logistic regression (baseline)
   - Embedding-based classifier (e.g. embedding + simple NN, or fine-tuned model)
3. Evaluate with appropriate metrics (precision, recall, F1 — explain your choice of macro/micro/weighted).
4. Show a confusion matrix or per-class breakdown.
5. Discuss: which classes are hardest to distinguish and why?

---

## 3. Embeddings & similarity search

Create embeddings for the job postings and implement a semantic search function.

1. Choose an embedding model and justify your choice.
2. Embed the preprocessed job descriptions (or appropriate chunks).
3. Index them for efficient retrieval (e.g. FAISS, or brute-force cosine similarity for this dataset size).
4. Implement a search function that takes a natural-language query and returns the top-k most relevant postings.

Some suitable models from the `sentence-transformers` library:

- `all-MiniLM-L6-v2` — fast, lightweight (384 dims), good general-purpose baseline
- `all-mpnet-base-v2` — higher quality, larger (768 dims)
- `multi-qa-MiniLM-L6-cos-v1` — optimised for semantic search / question-answering

Pick one (or compare two) and explain your reasoning.

Demonstrate your search with at least **3 example queries**, e.g.:
- "machine learning engineer with NLP experience"
- "entry-level data analyst with SQL"
- "remote software engineer working on cloud infrastructure"

For each result, show the job title, company, and a short excerpt showing why it's relevant.

---

## 4. Evaluation & reflection

Write a brief section (in the notebook or a short markdown cell) covering:

1. **Classification errors:** Pick 2–3 misclassified examples from Task 2. Explain why the model likely got them wrong.
2. **Retrieval quality:** How would you measure whether your search returns relevant results? Propose a method (you don't need to fully implement it, but show your thinking).
3. **Limitations:** What are the main weaknesses of your pipeline? What would you improve with more time?

---

# Deliverables

Submit a GitHub repository or zipped project containing:

- This notebook (or equivalent scripts) with your solution;
- Dependency specification (`requirements.txt` or `pyproject.toml`);
- Reproducible setup instructions;
- Description of any API keys required.

---

# Notes

- A UI is not required.
- You may use any open-source libraries (scikit-learn, sentence-transformers, spaCy, etc.).
- If you use a pre-trained model, explain why you chose it.
- We care about your reasoning and methodology, not just the final numbers.
- If you run out of time, submit what you have with a note on what you'd do next.

---
## Data files

The following data files are provided in the `data/` directory:

| File | Rows | Description |
|------|-----:|-------------|
| `sample_postings.csv` | 4,072 | Job postings with title, description, company, location, salary, experience level, etc. |
| `sample_companies.csv` | 1,979 | Company profiles with name, description, size, location, and URL |
| `sample_skills.csv` | 6,800 | Skill tags linked to job postings (job_id → skill_abr) |
| `sample_skill_lookup.csv` | 35 | Skill abbreviation → full skill name mapping |
| `sample_industries.csv` | 422 | Industry labels (industry_id, industry_name) |

### Key columns

**sample_postings.csv:** `job_id`, `company_name`, `title`, `description`, `role_category`, `location`, `company_id`, `formatted_work_type`, `formatted_experience_level`, `skills_desc`, `max_salary`, `min_salary`, `remote_allowed`

**sample_companies.csv:** `company_id`, `name`, `description`, `company_size`, `state`, `country`, `city`, `url`

**sample_skills.csv:** `job_id`, `skill_abr`

**sample_skill_lookup.csv:** `skill_abr`, `skill_name` — maps abbreviations (e.g. `ENGR`) to full names (e.g. `Engineering`)

**sample_industries.csv:** `industry_id`, `industry_name`

---
## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")

# Load sample data
postings = pd.read_csv(DATA_DIR / "sample_postings.csv", low_memory=False)
print(f"Postings: {len(postings):,}")
print(f"Columns: {list(postings.columns)}")

Postings: 4,072
Columns: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips', 'role_category']


In [2]:
# Load related tables
skills = pd.read_csv(DATA_DIR / "sample_skills.csv", low_memory=False) if (DATA_DIR / "sample_skills.csv").exists() else None
skill_lookup = pd.read_csv(DATA_DIR / "sample_skill_lookup.csv") if (DATA_DIR / "sample_skill_lookup.csv").exists() else None
companies = pd.read_csv(DATA_DIR / "sample_companies.csv", low_memory=False) if (DATA_DIR / "sample_companies.csv").exists() else None

if skills is not None:
    print(f"Skills entries: {len(skills):,}")
if skill_lookup is not None:
    print(f"Skill categories: {len(skill_lookup)}")
if companies is not None:
    print(f"Companies: {len(companies):,}")

Skills entries: 6,800
Skill categories: 35
Companies: 1,979


---
## Explore the data

In [3]:
postings.head(3)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips,role_category
0,133130219,NaN,Software Engineer,"Education Bachelor's degree in software, math,...",NaN,NaN,Los Angeles Metropolitan Area,NaN,1.0,NaN,...,1.713535e+12,NaN,0,FULL_TIME,NaN,NaN,NaN,NaN,NaN,Software Engineer
1,175485704,GOYT,Software Engineer,Job Description:GOYT is seeking a skilled and ...,NaN,NaN,"Denver, CO",76987056.0,273.0,NaN,...,1.713281e+12,NaN,0,PART_TIME,NaN,NaN,NaN,80202.0,8031.0,Software Engineer
2,2234533717,Ideando Inc,Full Stack Engineer,"Location: Remote\nCompany Overview:SkillFit, a...",NaN,NaN,United States,69611476.0,21.0,NaN,...,1.713493e+12,NaN,0,FULL_TIME,NaN,NaN,NaN,NaN,NaN,Software Engineer


In [4]:
# Role distribution
postings["title"].value_counts().head(20)

title
Software Engineer                   181
Senior Software Engineer            164
Data Analyst                        137
Data Engineer                        91
Full Stack Engineer                  58
DevOps Engineer                      57
Data Scientist                       56
Senior Data Engineer                 44
Cloud Engineer                       32
Embedded Software Engineer           30
Senior Data Analyst                  25
Machine Learning Engineer            23
Principal Software Engineer          21
Senior Data Scientist                21
Lead Software Engineer               20
Online Data Analyst                  20
Software Engineer in Test            20
Staff Software Engineer              19
Senior Machine Learning Engineer     19
Java Software Engineer               16
Name: count, dtype: int64

In [5]:
# Check description field
print(f"Postings with description: {postings['description'].notna().sum():,}")
print(f"\nSample description (first 500 chars):")
print(postings['description'].dropna().iloc[0][:500])

Postings with description: 4,072

Sample description (first 500 chars):
Education Bachelor's degree in software, math, or science required Job Skills Analytical skills, group work, knowledge of intended audience, understanding of different roles


---
## Your work starts here

Use the cells above as a starting point. Add new cells below to build your solution.